In [5]:
"""
MooVision Prediction Distribution Analysis
------------------------------------------
Reads all per-video prediction JSONs from a directory and produces
Altair charts saved to an HTML file.

Usage:
    python visualize_predictions.py --input_dir /path/to/prediction/jsons
                                    --output    predictions_analysis.html
"""

import json
import re
import argparse
from pathlib import Path

import pandas as pd
import altair as alt
import sys

# Get the project root (go up from notebook location)
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from config import ROOT_DIR

EVAL_PATH = ROOT_DIR / "results" / "evaluation"
BASELINE_MODEL_OUTPUT_DIR = ROOT_DIR / "data" / "results" / "metadata" / "baseline"
YOLO_MODEL_OUTPUT_DIR = ROOT_DIR / "results" / "metadata" / "random" / "yolo"
SEQ_MODEL_OUTPUT_DIR = ROOT_DIR / "results" / "metadata" / "random" / "seq-nms"

In [6]:
# Filename parsing
def _parse_filename(filename: str) -> dict:
    """
    Infer split name and model from a JSON filename.
 
    Expected format:
        evaluation_report_<split>_<model>.json
 
    Where model is one of: yolo, seq_nms.
    Everything between 'evaluation_report_' and the model suffix
    is treated as the split name.
 
    Parameters
    ----------
    filename : str
        JSON filename without directory prefix.
 
    Returns
    -------
    dict
        Keys: split, model. Returns {"split": filename, "model": "unknown"}
        if the pattern does not match.
    """
    stem = Path(filename).stem  # strip .json
 
    # Match model suffix — yolo or seq_nms at the end
    m = re.match(r"evaluation_report_(.+?)_(yolo|seq_nms)$", stem)
    if not m:
        return {"split": stem, "model": "unknown"}
 
    return {
        "split": m.group(1),
        "model": m.group(2),
    }
 
 
# Single file loader 
def _load_one(path: Path) -> dict:
    """
    Load one evaluation report JSON and extract summary metrics.
 
    Parameters
    ----------
    path : Path
        Path to a single evaluation report JSON.
 
    Returns
    -------
    dict
        Flat dict with split, model, and all key metrics.
    """
    with open(path) as f:
        data = json.load(f)
 
    meta   = _parse_filename(path.name)
    ev     = data.get("event_level", {})
    seq    = data.get("sequence_level", {})
    frame  = data.get("frame_level", {})
 
    return {
        "split":                meta["split"],
        "model":                meta["model"],
        "true_positives":       ev.get("true_positives",  0),
        "false_positives":      ev.get("false_positives", 0),
        "false_negatives":      ev.get("false_negatives", 0),
        "precision":            ev.get("precision",       0.0),
        "recall":               ev.get("recall",          0.0),
        "f1":                   ev.get("f1",              0.0),
        "f2":                   ev.get("f2",              0.0),
        "event_avg_bbox_iou":   ev.get("avg_bbox_iou",    0.0),
        "avg_temporal_iou":     seq.get("avg_temporal_iou", 0.0),
        "frame_avg_bbox_iou":   frame.get("avg_bbox_iou",   0.0),
    }
 
 
# Directory loader 
def load_evaluation_folder(folder: str | Path) -> pd.DataFrame:
    """
    Read all evaluation report JSONs from a folder and return a summary table.
 
    Input
    -----
      - folder : Directory containing evaluation report JSON files.
        Files are expected to follow the naming convention:
            evaluation_report_<split>_<model>.json
 
    Output
    ------
      - A DataFrame with one row per JSON file, sorted by split then model,
        containing event-level, sequence-level, and frame-level metrics.
 
    Parameters
    ----------
    folder : str or Path
        Path to the directory containing evaluation JSON files.
 
    Returns
    -------
    pd.DataFrame
        Columns: split, model, true_positives, false_positives,
        false_negatives, precision, recall, f1, f2,
        event_avg_bbox_iou, avg_temporal_iou, frame_avg_bbox_iou.
 
    Raises
    ------
    FileNotFoundError
        If the folder does not exist.
    ValueError
        If no JSON files are found in the folder.
    """
    folder = Path(folder)
 
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")
 
    json_files = sorted(folder.glob("*.json"))
    if not json_files:
        raise ValueError(f"No JSON files found in: {folder}")
 
    rows = [_load_one(jf) for jf in json_files]
    df = pd.DataFrame(rows).sort_values(["split", "model"]).reset_index(drop=True)
    return df

load_evaluation_folder(EVAL_PATH)

,split,model,true_positives,false_positives,false_negatives,precision,recall,f1,f2,event_avg_bbox_iou,avg_temporal_iou,frame_avg_bbox_iou
0,pen_based_pen_2,seq_nms,2,174,1098,0.0114,0.0018,0.0031,0.0022,0.0542,0.6038,0.3672
1,pen_based_pen_2,yolo,2,28,1098,0.0667,0.0018,0.0035,0.0023,0.4423,0.6888,0.2961
2,pen_based_pen_3,seq_nms,1,72,1099,0.0137,0.0009,0.0017,0.0011,0.0000,0.7092,0.5336
3,pen_based_pen_3,yolo,0,8,1100,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3722
4,pen_based_pen_5,seq_nms,2,173,1098,0.0114,0.0018,0.0031,0.0022,0.0000,0.7860,0.4537
5,pen_based_pen_5,yolo,0,18,1100,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2999
6,period_based_POSTWEAN,seq_nms,9,193,1091,0.0446,0.0082,0.0138,0.0098,0.1140,0.5959,0.4585
7,period_based_POSTWEAN,yolo,3,27,1097,0.1000,0.0027,0.0053,0.0034,0.1278,0.6442,0.2972
8,period_based_WEAN,seq_nms,5,569,1095,0.0087,0.0045,0.0060,0.0050,0.0000,0.6195,0.4188
9,period_based_WEAN,yolo,1,18,1099,0.0526,0.0009,0.0018,0.0011,0.0000,0.6482,0.2494


In [ ]:
def load_predictions(input_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
        video_df  – one row per video
        event_df  – one row per event (exploded)
    """
    video_rows, event_rows = [], []

    for path in sorted(Path(input_dir).glob("*.json")):
        with open(path) as f:
            data = json.load(f)

        vid = data["identifier"]
        pen = _extract_pen(data.get("video_path", ""))
        stage = _extract_stage(data.get("video_path", ""))

        video_rows.append({
            "video":                   vid,
            "pen":                     pen,
            "weaning_stage":           stage,
            "total_duration_sec":      data.get("total_duration_sec"),
            "cross_sucking_detected":  data.get("cross_sucking_detected", False),
            "num_events":              data.get("num_events", 0),
        })

        for i, ev in enumerate(data.get("events", [])):
            event_rows.append({
                "video":           vid,
                "pen":             pen,
                "weaning_stage":   stage,
                "event_idx":       i,
                "start_sec":       ev["start_sec"],
                "end_sec":         ev["end_sec"],
                "duration_sec":    ev["duration_sec"],
                "avg_confidence":  ev["avg_confidence"],
            })

    video_df = pd.DataFrame(video_rows)
    event_df = pd.DataFrame(event_rows) if event_rows else pd.DataFrame(
        columns=["video", "pen", "weaning_stage", "event_idx",
                 "start_sec", "end_sec", "duration_sec", "avg_confidence"]
    )
    return video_df, event_df


def _extract_pen(video_path: str) -> str:
    """Best-effort pen extraction from path string."""
    import re
    m = re.search(r"Pen\s*(\d+)", video_path, re.IGNORECASE)
    return f"Pen {m.group(1)}" if m else "Unknown"


def _extract_stage(video_path: str) -> str:
    for stage in ("PREWEANING", "WEANING", "POSTWEANING"):
        if stage.lower() in video_path.lower():
            return stage
    return "Unknown"

video_df, event_df = load_predictions(SEQ_MODEL_OUTPUT_DIR)

In [ ]:
event_df

In [ ]:
def chart_confidence_distribution(event_df: pd.DataFrame) -> alt.Chart:
    """Histogram of avg_confidence across all events."""
    return (
        alt.Chart(event_df, title="Confidence Score Distribution (all events)")
        .mark_bar(opacity=0.8, color="steelblue")
        .encode(
            alt.X("avg_confidence:Q",
                  bin=alt.Bin(step=0.02),
                  title="Avg Confidence"),
            alt.Y("count():Q", title="# Events"),
            tooltip=["count():Q"],
        )
        .properties(width=500, height=280)
    )

chart_confidence_distribution(event_df)

In [ ]:
def chart_confidence_by_pen(event_df: pd.DataFrame) -> alt.Chart:
    """Overlapping confidence histograms faceted by pen."""
    return (
        alt.Chart(event_df, title="Confidence Distribution by Pen")
        .mark_bar(opacity=0.8,color="steelblue")
        .encode(
            alt.X("avg_confidence:Q", bin=alt.Bin(step=0.02), title="Avg Confidence"),
            alt.Y("count():Q", title="# Events"),
            alt.Color("pen:N", title="Pen"),
            tooltip=["pen:N", "count():Q"],
        )
        .properties(width=500, height=280)
    )
chart_confidence_by_pen(event_df)

In [ ]:
def chart_events_per_video(video_df: pd.DataFrame) -> alt.Chart:
    """Bar chart: number of predicted events per video, coloured by pen."""
    df = video_df.sort_values("num_events", ascending=False).copy()
    df["short_name"] = df["video"].str.replace(r"\.mp4$", "", regex=True)
 
    return (
        alt.Chart(df, title="Predicted Events per Video")
        .mark_bar()
        .encode(
            alt.X("short_name:N",
                  sort="-y",
                  title="Video",
                  axis=alt.Axis(labelAngle=-45, labelLimit=200)),
            alt.Y("num_events:Q", title="# Predicted Events"),
            alt.Color("pen:N", title="Pen"),
            tooltip=["short_name:N", "pen:N", "weaning_stage:N", "num_events:Q"],
        )
        .properties(width=700, height=300)
    )
chart_events_per_video(video_df)

In [ ]:
def chart_event_duration_distribution(event_df: pd.DataFrame) -> alt.Chart:
    """Histogram of event durations in seconds."""
    return (
        alt.Chart(event_df, title="Event Duration Distribution")
        .mark_bar(opacity=0.8, color="#F58518")
        .encode(
            alt.X("duration_sec:Q",
                  bin=alt.Bin(maxbins=30),
                  title="Duration (s)"),
            alt.Y("count():Q", title="# Events"),
            tooltip=["count():Q"],
        )
        .properties(width=500, height=280)
    )
chart_event_duration_distribution(event_df)

In [ ]:
def chart_temporal_density(event_df: pd.DataFrame) -> alt.Chart:
    """
    Scatter: event start time vs video (y-axis), sized by duration, coloured by confidence.
    Shows where within each video events are firing.
    """
    df = event_df.copy()
    df["short_name"] = df["video"].str.replace(r"\.mp4$", "", regex=True)
 
    return (
        alt.Chart(df, title="Temporal Density of Events Within Videos")
        .mark_point(filled=True, opacity=0.75)
        .encode(
            alt.X("start_sec:Q", title="Start Time (s)"),
            alt.Y("short_name:N", title="Video", sort="-x"),
            alt.Size("duration_sec:Q", title="Duration (s)",
                     scale=alt.Scale(range=[20, 300])),
            alt.Color("avg_confidence:Q",
                      scale=alt.Scale(scheme="viridis"),
                      title="Confidence"),
            tooltip=["short_name:N", "start_sec:Q", "end_sec:Q",
                     "duration_sec:Q", "avg_confidence:Q", "pen:N","weaning_stage:N"],
        )
        .properties(width=700, height=max(300, len(df["video"].unique()) * 22))
    )
chart_temporal_density(event_df)

In [ ]:
def chart_confidence_vs_duration(event_df: pd.DataFrame) -> alt.Chart:
    """Scatter: confidence vs duration, coloured by pen."""
    df = event_df.copy()
    df["short_name"] = df["video"].str.replace(r"\.mp4$", "", regex=True)
 
    return (
        alt.Chart(df, title="Confidence vs Event Duration")
        .mark_point(filled=True, opacity=0.7, size=60)
        .encode(
            alt.X("duration_sec:Q", title="Duration (s)"),
            alt.Y("avg_confidence:Q", title="Avg Confidence"),
            alt.Color("pen:N", title="Pen"),
            tooltip=["short_name:N", "pen:N", "weaning_stage:N",
                     "duration_sec:Q", "avg_confidence:Q"],
        )
        .properties(width=500, height=300)
    )
chart_confidence_vs_duration(event_df)